## 0) paths


In [1]:
import pandas as pd
import geopandas as gpd
import trackintel as ti
from eval import get_score
from pathlib import Path

CWD = Path.cwd().resolve()

if (CWD / "sps").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "sps").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(f"Cannot find project root containing 'sps' from {CWD}")

SPS_DIR = PROJECT_ROOT / "sps"
IN_PARQUET = SPS_DIR / "noised_trajectory.parquet"
GT_PARQUET = SPS_DIR / "naive_sps.parquet"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("IN_PARQUET   =", IN_PARQUET)
print("GT_PARQUET   =", GT_PARQUET)


PROJECT_ROOT = D:\staypoint_ptpe
IN_PARQUET   = D:\staypoint_ptpe\sps\noised_trajectory.parquet
GT_PARQUET   = D:\staypoint_ptpe\sps\naive_sps.parquet


## 1) parameters


In [2]:
PARAM_GROUPS = [
    {"name": "current", "DIST_M": 84.0, "TIME_MIN": 7.0, "GAP_MIN": 60.0},
    {"name": "baseline", "DIST_M": 50.0, "TIME_MIN": 5.0, "GAP_MIN": 25.0},
]

# TIMEZONE = "UTC"   # change to "US/Eastern" or another timezone you want
TIMEZONE = "US/Eastern"


## 2) helpers


In [3]:
def ensure_tz(df, col, tz=TIMEZONE):
    s = pd.to_datetime(df[col], errors="coerce")
    if getattr(s.dt, "tz", None) is None:
        s = s.dt.tz_localize(tz)
    else:
        s = s.dt.tz_convert(tz)
    df[col] = s
    return df


def sp_to_eval_df(sp, tz=TIMEZONE):
    sp_out = sp.copy()

    sp_out["latitude"] = sp_out.geometry.y
    sp_out["longitude"] = sp_out.geometry.x
    sp_out["agent_id"] = sp_out["user_id"]

    start_col = "started_at" if "started_at" in sp_out.columns else "arrive_time"
    end_col = "finished_at" if "finished_at" in sp_out.columns else "leave_time"

    sp_out["arrive_time"] = pd.to_datetime(sp_out[start_col], errors="coerce")
    sp_out["leave_time"] = pd.to_datetime(sp_out[end_col], errors="coerce")

    sp_out = sp_out[["agent_id", "latitude", "longitude", "arrive_time", "leave_time"]].copy()
    sp_out = sp_out.dropna(subset=["agent_id", "latitude", "longitude", "arrive_time", "leave_time"])

    sp_out = ensure_tz(sp_out, "arrive_time", tz=tz)
    sp_out = ensure_tz(sp_out, "leave_time", tz=tz)
    return sp_out


## 3) load noisy trajectory


In [4]:
df = pd.read_parquet(IN_PARQUET).copy()

df["tracked_at"] = pd.to_datetime(df["time"], errors="coerce")
if getattr(df["tracked_at"].dt, "tz", None) is None:
    df["tracked_at"] = df["tracked_at"].dt.tz_localize(TIMEZONE)
else:
    df["tracked_at"] = df["tracked_at"].dt.tz_convert(TIMEZONE)

df["user_id"] = df["agent_id"]
df = df.dropna(subset=["user_id", "tracked_at", "n_lat", "n_lon"])
df = df.sort_values(["user_id", "tracked_at"])

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["n_lon"], df["n_lat"]),
    crs="EPSG:4326",
)

pfs = ti.io.read_positionfixes_gpd(
    gdf,
    tracked_at="tracked_at",
    user_id="user_id",
    tz=TIMEZONE
)


## 4) load ground truth


In [5]:
gt = pd.read_parquet(GT_PARQUET).rename(columns={
    "user_id": "agent_id",
    "lat": "latitude",
    "lon": "longitude",
    "startTime": "arrive_time",
    "endTime": "leave_time",
})

gt = gt[["agent_id", "latitude", "longitude", "arrive_time", "leave_time"]].copy()
gt = ensure_tz(gt, "arrive_time", tz=TIMEZONE)
gt = ensure_tz(gt, "leave_time", tz=TIMEZONE)
gt = gt.dropna(subset=["agent_id", "latitude", "longitude", "arrive_time", "leave_time"])


## 5) generate staypoints + score for each parameter group


In [6]:
all_results = []

print("TIMEZONE =", TIMEZONE)

for params in PARAM_GROUPS:
    run_name = params["name"]
    dist_m = params["DIST_M"]
    time_min = params["TIME_MIN"]
    gap_min = params["GAP_MIN"]

    print("\n" + "=" * 60)
    print(f"Run: {run_name}")
    print(f"DIST_M={dist_m}, TIME_MIN={time_min}, GAP_MIN={gap_min}")

    _, sp = pfs.generate_staypoints(
        method="sliding",
        distance_metric="haversine",
        dist_threshold=dist_m,
        time_threshold=time_min,
        gap_threshold=gap_min,
        exclude_duplicate_pfs=True,
    )

    pred_eval = sp_to_eval_df(sp, tz=TIMEZONE)

    print("Number of predicted staypoints:", len(pred_eval))
    print(pred_eval.head())

    # A. reproduce current notebook behavior exactly
    score_same_as_notebook = get_score(
        ground_truth_df=gt,
        calculated_df=pred_eval
    )
    print("\nScore using current notebook logic (default eval.py r=0.001, t=5):")
    print(score_same_as_notebook)

    row = {
        "name": run_name,
        "DIST_M": dist_m,
        "TIME_MIN": time_min,
        "GAP_MIN": gap_min,
        "n_pred": len(pred_eval),
    }
    row.update(score_same_as_notebook)
    all_results.append(row)


TIMEZONE = US/Eastern

Run: current
DIST_M=84.0, TIME_MIN=7.0, GAP_MIN=60.0
Number of predicted staypoints: 65118
    agent_id   latitude  longitude               arrive_time  \
id                                                             
0          0  33.770987 -84.419322 2024-01-01 00:00:00-05:00   
1          0  33.751780 -84.393275 2024-01-01 08:00:00-05:00   
2          0  33.755122 -84.370098 2024-01-01 13:22:00-05:00   
3          0  33.751793 -84.393283 2024-01-01 14:20:00-05:00   
4          0  33.759423 -84.350699 2024-01-01 19:34:00-05:00   

                  leave_time  
id                            
0  2024-01-01 07:07:00-05:00  
1  2024-01-01 12:44:00-05:00  
2  2024-01-01 13:43:00-05:00  
3  2024-01-01 17:00:00-05:00  
4  2024-01-01 21:29:00-05:00  
[2026-03-22 12:21:19]: Calculating recall score...
[2026-03-22 12:22:18]: Calculating precision score...

Score using current notebook logic (default eval.py r=0.001, t=5):
{'precision': 0.9824472496084032, 'recall': 0.9

## 6) final summary


In [7]:
summary_df = pd.DataFrame(all_results)

print("\n" + "=" * 60)
print("Final summary:")
print(summary_df)



Final summary:
       name  DIST_M  TIME_MIN  GAP_MIN  n_pred  precision    recall        f1  \
0   current    84.0       7.0     60.0   65118   0.982447  0.971187  0.976785   
1  baseline    50.0       5.0     25.0  118638   0.445304  0.801998  0.572649   

         f2  
0  0.973418  
1  0.691257  
